In [0]:
# ─────────────────────────────────────────────────────
# common_configs.py
# Dynamic ADLS connection via Key Vault + Service Principal
# No hardcoding anywhere
# ─────────────────────────────────────────────────────

SECRET_SCOPE = "azure-kv-scope"

try:
    # ── Fetch secrets from Key Vault ──────────────
    client_id = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="client-id"
    )

    tenant_id = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="tenant-id"
    )

    client_secret = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="client-secret"
    )

    storage_account = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="storage-account"
    )

    domain = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="pipeline-domain"          # ingestion_logs
    )

    catalog_name = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="unity-catalog-name"       # bib_catalog
    )

    # ── Gemini API Key from Key Vault ──────
    gemini_api_key = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="gemini-api-key"
    )

    gemini_model_name = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="gemini-model-name"
    )   

    # ── Configure Spark for OAuth ─────────────────
    spark.conf.set(
        f"fs.azure.account.auth.type"
        f".{storage_account}.dfs.core.windows.net",
        "OAuth"
    )

    spark.conf.set(
        f"fs.azure.account.oauth.provider.type"
        f".{storage_account}.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2"
        ".ClientCredsTokenProvider"
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.id"
        f".{storage_account}.dfs.core.windows.net",
        client_id
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.secret"
        f".{storage_account}.dfs.core.windows.net",
        client_secret
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.endpoint"
        f".{storage_account}.dfs.core.windows.net",
        f"https://login.microsoftonline.com/"
        f"{tenant_id}/oauth2/token"
    )

    # ── Validate connection ───────────────────────
    test_path = (
        f"abfss://dev@{storage_account}"
        f".dfs.core.windows.net/"
    )
    dbutils.fs.ls(test_path)

    print("================================")
    print("BIB ADLS Connection Successful")
    print(f"   Storage Account : {storage_account}")
    print(f"   Catalog         : {catalog_name}")
    print(f"   Secret Scope    : {SECRET_SCOPE}")
    print("================================")

except Exception as e:
    print("================================")
    print("BIB ADLS Connection Failed")
    print(str(e))
    print("================================")
    
    raise e


# ─────────────────────────────────────────────────────
# Path helper functions
# Used by all notebooks — bronze, silver, gold, generate
# ─────────────────────────────────────────────────────

def get_adls_path(environment, layer, sub_path=""):
    base = (
        f"abfss://{environment}"
        f"@{storage_account}"
        f".dfs.core.windows.net/{layer}"
    )
    return f"{base}/{sub_path}" if sub_path else base


def get_raw_path(
    environment, raw_path,
    year, month, day, is_daily
):
    if is_daily == "Y":
        return get_adls_path(
            environment, "raw",
            f"{raw_path}/"
            f"year={year}/month={month}/day={day}/"
        )
    return get_adls_path(
        environment, "raw", f"{raw_path}/"
    )